In [ ]:
# ============================================================
# Task 4 — Regression: Revenue Forecasting
# Predict MonthlyCharges using remaining features
# ============================================================
import sys
import os
import time
import warnings
warnings.filterwarnings("ignore")
sys.path.append(os.path.abspath(".."))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR

from src.utils import save_figure

sns.set_style("whitegrid")


# ============================================================
# Step 1 — Load and Prepare Regression Dataset
# ============================================================
df = pd.read_csv("../data/processed/telco_cleaned.csv")

# Target
y = df["monthlycharges"]

# Features
X = df.drop(columns=["customerid", "monthlycharges"])

# Encode categorical columns
categorical_cols = X.select_dtypes(include=["object"]).columns

for col in categorical_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])

# Train / Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scale features
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X_train.columns
)

X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=X_test.columns
)

# Store results
results = []


# ============================================================
# Helper Function — Train, Tune, Evaluate, Save
# ============================================================
def train_and_evaluate(model_name, model, param_grid, Xtr, Xte, ytr, yte):
    start = time.time()

    grid = GridSearchCV(
        estimator=model,
        param_grid=param_grid,
        cv=5,
        scoring="r2",
        n_jobs=-1
    )

    grid.fit(Xtr, ytr)
    best_model = grid.best_estimator_

    train_time = time.time() - start

    y_pred = best_model.predict(Xte)

    mae = mean_absolute_error(yte, y_pred)
    mse = mean_squared_error(yte, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(yte, y_pred)

    n = Xte.shape[0]
    p = Xte.shape[1]
    adj_r2 = 1 - ((1 - r2) * (n - 1) / (n - p - 1))

    results.append({
        "Model": model_name,
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "R2": r2,
        "Adj_R2": adj_r2,
        "Training_Time": train_time
    })

    # Actual vs Predicted Plot
    plt.figure(figsize=(6, 5))
    plt.scatter(yte, y_pred, alpha=0.5)
    plt.plot([yte.min(), yte.max()], [yte.min(), yte.max()], "r--")
    plt.xlabel("Actual MonthlyCharges")
    plt.ylabel("Predicted MonthlyCharges")
    plt.title(f"{model_name} - Actual vs Predicted")
    save_figure(f"../reports/figures/{model_name.lower().replace(' ', '_')}_actual_vs_predicted.png")
    plt.show()

    print(f"{model_name} completed.")
    print("Best Params:", grid.best_params_)
    print(f"R2 Score: {r2:.4f}")
    print("-" * 60)

    return best_model, y_pred


# ============================================================
# Step 2 — Train Models One by One
# ============================================================

# 1. Linear Regression
lr_model, lr_pred = train_and_evaluate(
    "Linear Regression",
    LinearRegression(),
    {"fit_intercept": [True, False]},
    X_train_scaled, X_test_scaled, y_train, y_test
)

# 2. Ridge Regression
ridge_model, ridge_pred = train_and_evaluate(
    "Ridge",
    Ridge(),
    {"alpha": [0.01, 0.1, 1, 10]},
    X_train_scaled, X_test_scaled, y_train, y_test
)

# 3. Lasso Regression
lasso_model, lasso_pred = train_and_evaluate(
    "Lasso",
    Lasso(max_iter=5000),
    {"alpha": [0.001, 0.01, 0.1, 1]},
    X_train_scaled, X_test_scaled, y_train, y_test
)

# 4. ElasticNet
elastic_model, elastic_pred = train_and_evaluate(
    "ElasticNet",
    ElasticNet(max_iter=5000),
    {
        "alpha": [0.001, 0.01, 0.1, 1],
        "l1_ratio": [0.2, 0.5, 0.8]
    },
    X_train_scaled, X_test_scaled, y_train, y_test
)

# 5. Decision Tree Regressor
dt_model, dt_pred = train_and_evaluate(
    "Decision Tree",
    DecisionTreeRegressor(random_state=42),
    {
        "max_depth": [3, 5, 10, None],
        "min_samples_split": [2, 5, 10]
    },
    X_train, X_test, y_train, y_test
)

# 6. Random Forest Regressor
rf_model, rf_pred = train_and_evaluate(
    "Random Forest",
    RandomForestRegressor(random_state=42),
    {
        "n_estimators": [100, 200],
        "max_depth": [5, 10, None]
    },
    X_train, X_test, y_train, y_test
)

# 7. Support Vector Regressor
svr_model, svr_pred = train_and_evaluate(
    "SVR",
    SVR(),
    {
        "C": [0.1, 1, 10],
        "kernel": ["linear", "rbf"]
    },
    X_train_scaled, X_test_scaled, y_train, y_test
)


# ============================================================
# Step 3 — Final Comparison Table
# ============================================================
results_df = pd.DataFrame(results)
results_df = results_df.sort_values("R2", ascending=False)

display(results_df)

results_df.to_csv("../artifacts/regression_results.csv", index=False)


# ============================================================
# Step 4 — Residual Analysis (Best Model)
# ============================================================
best_model_name = results_df.iloc[0]["Model"]
print("Best Model:", best_model_name)

best_predictions = {
    "Linear Regression": lr_pred,
    "Ridge": ridge_pred,
    "Lasso": lasso_pred,
    "ElasticNet": elastic_pred,
    "Decision Tree": dt_pred,
    "Random Forest": rf_pred,
    "SVR": svr_pred
}

best_pred = best_predictions[best_model_name]
residuals = y_test - best_pred

plt.figure(figsize=(8, 5))
sns.histplot(residuals, kde=True)
plt.title(f"{best_model_name} - Residual Distribution")
plt.xlabel("Residuals")
save_figure("../reports/figures/best_model_residual_distribution.png")
plt.show()


# ============================================================
# Step 5 — Coefficient Analysis (Linear Models)
# ============================================================
linear_models = {
    "Linear Regression": lr_model,
    "Ridge": ridge_model,
    "Lasso": lasso_model,
    "ElasticNet": elastic_model
}

for name, model in linear_models.items():
    coef_df = pd.DataFrame({
        "Feature": X_train_scaled.columns,
        "Coefficient": model.coef_
    }).sort_values("Coefficient", key=abs, ascending=False)

    print(f"\n{name} - Top Coefficients")
    display(coef_df.head(10))

ModuleNotFoundError: No module named 'src'